In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [0]:
builder = (SparkSession.builder
           .appName("change-data-feed-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

### Create Bronze Table 
(as an appendOnly table)

In [0]:
%%sparksql
CREATE OR REPLACE TABLE default.movie_and_show_titles_cdf (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING 
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/movie_and_show_titles_cdf'
TBLPROPERTIES (delta.enableChangeDataFeed = true, medallionLevel = 'bronze');

#### Initial Load of Bronze Table

In [0]:
# Read CSV file into a DataFrame
df = (spark.read
      .format("csv")
      .option("header", "true")
      .load("../data/netflix_titles.csv"));
df.write.format("delta").mode("append").saveAsTable("default.movie_and_show_titles_cdf")

In [0]:
%%sparksql
CREATE OR REPLACE TABLE default.movie_and_show_titles_cleansed (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING 
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/movie_and_show_titles_cleansed'
TBLPROPERTIES (delta.enableChangeDataFeed = true, medallionLevel = 'silver'
, updatedFromTable= 'default.movie_and_show_titles_cdf', updatedFromTableVersion= '-1');

In [0]:
#get the value of the last Updated Version from the silver table
lastUpdateVersion = int(spark.sql("SHOW TBLPROPERTIES default.movie_and_show_titles_cleansed ('updatedFromTableVersion')").first()["value"])+1
lastUpdateVersion

In [0]:
#get the value of the last Updated Version from the silver table
latestVersion = spark.sql("DESCRIBE HISTORY default.movie_and_show_titles_cdf").first()["version"]
latestVersion

#### Create temp view of chnage to bronze table since last load of silver

In [0]:
%%sparksql
CREATE OR REPLACE TEMPORARY VIEW bronzeTable_latest_version as
SELECT * FROM (
    SELECT *, 
        RANK() OVER (
        PARTITION BY (lower(type), lower(title), lower(director), date_added) 
        ORDER BY _commit_version DESC) as rank  
    FROM table_changes('default.movie_and_show_titles_cdf',{lastUpdateVersion},{latestVersion})
    WHERE type IS NOT NULL AND title IS NOT NULL AND director IS NOT NULL AND  _change_type !='update_preimage'
)
WHERE rank = 1;

### Merge Change Data into Silver

In [0]:
%%sparksql 
MERGE INTO default.movie_and_show_titles_cleansed t 
USING bronzeTable_latest_version s 
ON lower(t.type) = lower(s.type)
AND lower(t.title) = lower(s.title)
AND lower(t.director) = lower(s.director)
AND t.date_added = s.date_added
WHEN MATCHED AND s._change_type='update_postimage' OR s._change_type='update_postimage' THEN UPDATE SET *
WHEN MATCHED AND s._change_type='delete' THEN DELETE
WHEN NOT MATCHED AND s._change_type='insert' THEN INSERT *

In [0]:
%%sparksql
ALTER TABLE default.movie_and_show_titles_cleansed SET TBLPROPERTIES(updatedFromTableVersion = {latestVersion});

In [0]:
%%sparksql
DROP VIEW bronzeTable_latest_version

### Update Bronze Table

In [0]:
%%sparksql
DELETE FROM default.movie_and_show_titles_cdf WHERE country is NULL

In [0]:
%%sparksql
UPDATE default.movie_and_show_titles_cdf SET director = '' WHERE director is NULL

### Propogate Changes from Bronze to Silver

In [0]:
#get the value of the last Updated Version from the silver table
lastUpdateVersion = spark.sql("SHOW TBLPROPERTIES default.movie_and_show_titles_cleansed ('updatedFromTableVersion')").first()["value"]
lastUpdateVersion

In [0]:
#get the value of the last Updated Version from the silver table
latestVersion = spark.sql("DESCRIBE HISTORY default.movie_and_show_titles_cdf").first()["version"]
latestVersion

In [0]:
%%sparksql
CREATE OR REPLACE TEMPORARY VIEW bronzeTable_latest_version as
SELECT * FROM (
    SELECT *, 
        RANK() OVER (
        PARTITION BY (lower(type), lower(title), lower(director), date_added) 
        ORDER BY _commit_version DESC) as rank  
    FROM table_changes('default.movie_and_show_titles_cdf',{lastUpdateVersion},{latestVersion})
    WHERE type IS NOT NULL AND title IS NOT NULL AND director IS NOT NULL AND  _change_type !='update_preimage'
)
WHERE rank = 1;

In [0]:
%%sparksql
SELECT _change_type, COUNT(*) FROM bronzeTable_latest_version GROUP BY _change_type

In [0]:
%%sparksql 
MERGE INTO default.movie_and_show_titles_cleansed t 
USING bronzeTable_latest_version s 
ON lower(t.type) = lower(s.type)
AND lower(t.title) = lower(s.title)
AND lower(t.director) = lower(s.director)
AND t.date_added = s.date_added
WHEN MATCHED AND s._change_type='update_postimage' THEN UPDATE SET *
WHEN MATCHED AND s._change_type='delete' THEN DELETE
WHEN NOT MATCHED AND s._change_type='insert' THEN INSERT *

In [0]:
%%sparksql
ALTER TABLE default.movie_and_show_titles_cleansed SET TBLPROPERTIES(updatedFromTableVersion = {latestVersion});

In [0]:
spark.stop()